# cytozip `dataloader` demo

Window-by-window loading of single-cell cytosine methylation data (deep-learning DataLoader use case).

- reference: `~/Ref/hg38/hg38_with_chrL.allc.cz`
- CG index: `~/Ref/hg38/hg38_with_chrL.CGN.cz`
- single-cell cz directory: `~/Projects/test_cytozip/benchmark/cz/` (first 500 cells)
- window size: `binsize = 2000`

Environment: conda `m3c` (cytozip installed from source, with the compiled Cython accelerator).

> Note: `iter_windows` / `load_chrom` materialize the whole chromosome matrix `(n_cells, n_sites)` in RAM. For chr1 (~4.75M CG sites) with 500 cells at `uint8` this is ~4.75 GB (mc + cov).

In [1]:
import os, glob, time
import numpy as np

import cytozip
from cytozip.dataloader import CzWindowLoader, resolve_cell_ids
print("cytozip:", cytozip.__file__)

REFERENCE = os.path.expanduser("~/Ref/hg38/hg38_with_chrL.allc.cz")
CGN_INDEX = os.path.expanduser("~/Ref/hg38/hg38_with_chrL.CGN.cz")
CELLS_DIR = os.path.expanduser("~/Projects/test_cytozip/benchmark/cz")

all_cells = sorted(glob.glob(os.path.join(CELLS_DIR, "*.cz")))
print(f"total single cells in directory: {len(all_cells)}")

# Test with the first 500 cells; use uint8 (mc/cov <= 255) so the
# (n_cells, n_sites) matrices are ~4x smaller than int32.
cells = all_cells[:500]
DTYPE = np.uint8
print(f"using {len(cells)} cells, dtype={np.dtype(DTYPE).name}")

cytozip: /home/x-wding2/Software/conda/m3c/lib/python3.10/site-packages/cytozip/__init__.py
total single cells in directory: 2000
using 500 cells, dtype=uint8


## 1. Build the loader (opens reference / index / all cells once)

In [9]:
# cache_chroms=1: keep the current chromosome's decompressed matrix in RAM so
# repeated iter_windows / load_chrom calls for the same chrom reuse it (now the
# default, passed here explicitly for the already-installed build).
loader = CzWindowLoader(reference=REFERENCE, cells=cells, index=CGN_INDEX,
                        dtype=DTYPE, jobs=16, cache_chroms=1)

print("num chromosomes:", len(loader.chroms))
print("first 5 chromosomes:", loader.chroms[:5])
print("num cells:", len(loader.cell_ids))
print("first 3 cell_ids:", loader.cell_ids[:3])

num chromosomes: 26
first 5 chromosomes: ['chr1', 'chr10', 'chr11', 'chr12', 'chr13']
num cells: 500
first 3 cell_ids: ['UWA7648_CX1819_NAC_1_P1-1-I3-A1', 'UWA7648_CX1819_NAC_1_P1-1-I3-A10', 'UWA7648_CX1819_NAC_1_P1-1-I3-A11']


## 2. Iterate chr1 window by window (binsize=2 kb, CG sites only)

Each `Window` holds `chrom, start, end, pos, mc, cov`; `mc`/`cov` have shape `(n_cells, n_sites_in_window)`, with row order = `loader.cell_ids`.

In [10]:
CHROM = "chr1"
BINSIZE = 2000

for i, w in enumerate(loader.iter_windows(CHROM, binsize=BINSIZE)):
    print(f"bin {i}: {w.chrom}:{w.start:,}-{w.end:,}  mc{w.mc.shape} cov{w.cov.shape}  sites={w.pos.shape[0]}")
    if i >= 4:
        break

bin 0: chr1:10,000-12,000  mc(500, 266) cov(500, 266)  sites=266
bin 1: chr1:12,000-14,000  mc(500, 74) cov(500, 74)  sites=74
bin 2: chr1:14,000-16,000  mc(500, 100) cov(500, 100)  sites=100
bin 3: chr1:16,000-18,000  mc(500, 89) cov(500, 89)  sites=89
bin 4: chr1:18,000-20,000  mc(500, 85) cov(500, 85)  sites=85


## 3. Time to fetch the first 10 bins

Bin 0 pays the one-time cost of decompressing all of chr1 across every cell; later bins are just in-RAM slices.

In [11]:
t0 = time.perf_counter()
bins = []
for w in loader.iter_windows(CHROM, binsize=BINSIZE):
    bins.append(w)
    if len(bins) >= 10:
        break
elapsed_ms = (time.perf_counter() - t0) * 1e3
print(f"time to fetch first {len(bins)} bins: {elapsed_ms:.1f} ms")
print(f"bin0: {bins[0].mc.shape[0]} cells x {bins[0].mc.shape[1]} sites")

time to fetch first 10 bins: 587.2 ms
bin0: 500 cells x 266 sites


In [16]:
bins

[Window(chrom='chr1', start=10000, end=12000, pos=array([10469, 10470, 10471, 10472, 10484, 10485, 10489, 10490, 10493,
        10494, 10497, 10498, 10525, 10526, 10542, 10543, 10563, 10564,
        10571, 10572, 10577, 10578, 10579, 10580, 10589, 10590, 10609,
        10610, 10617, 10618, 10620, 10621, 10631, 10632, 10633, 10634,
        10636, 10637, 10638, 10639, 10641, 10642, 10644, 10645, 10650,
        10651, 10660, 10661, 10662, 10663, 10665, 10666, 10667, 10668,
        10670, 10671, 10673, 10674, 10679, 10680, 10689, 10690, 10691,
        10692, 10694, 10695, 10696, 10697, 10699, 10700, 10702, 10703,
        10708, 10709, 10718, 10719, 10720, 10721, 10723, 10724, 10725,
        10726, 10728, 10729, 10731, 10732, 10737, 10738, 10747, 10748,
        10749, 10750, 10752, 10753, 10754, 10755, 10757, 10758, 10760,
        10761, 10766, 10767, 10776, 10777, 10778, 10779, 10781, 10782,
        10783, 10784, 10786, 10787, 10789, 10790, 10795, 10796, 10811,
        10812, 10813, 10814,

## 4. Compute per-cell methylation level (mCG) for one window

In [13]:
w = bins[0]
cov = w.cov.astype(np.float32)
frac = np.where(cov > 0, w.mc / np.clip(cov, 1, None), np.nan)
per_cell = np.nanmean(frac, axis=1)  # mean mCG per cell in this window (ignoring uncovered sites)

print(f"window {w.chrom}:{w.start:,}-{w.end:,}  shape={frac.shape}")
for cid, m in zip(loader.cell_ids[:5], per_cell[:5]):
    print(f"{cid}: mCG={m:.3f}")

window chr1:10,000-12,000  shape=(500, 266)
UWA7648_CX1819_NAC_1_P1-1-I3-A1: mCG=0.850
UWA7648_CX1819_NAC_1_P1-1-I3-A10: mCG=nan
UWA7648_CX1819_NAC_1_P1-1-I3-A11: mCG=nan
UWA7648_CX1819_NAC_1_P1-1-I3-A12: mCG=nan
UWA7648_CX1819_NAC_1_P1-1-I3-A14: mCG=nan


/tmp/ipykernel_28610/1592952354.py:4: RuntimeWarning: Mean of empty slice
  per_cell = np.nanmean(frac, axis=1)  # mean mCG per cell in this window (ignoring uncovered sites)


## 5. Whole-chromosome matrix via `load_chrom`

In [12]:
# Whole-chromosome matrix for all loaded cells (reuses the loader's open readers).
pos, mc, cov = loader.load_chrom(CHROM)
print("chr1 CG sites:", f"{pos.shape[0]:,}", " matrix:", mc.shape, mc.dtype,
      f"({len(loader.cell_ids)} cells)")

chr1 CG sites: 4,750,318  matrix: (500, 4750318) uint8 (500 cells)


## 6. (Optional) Use as a PyTorch `IterableDataset`

In [14]:
try:
    import torch
    from cytozip.dataloader import CzWindowDataset
    # small subset just to show tensor output
    ds = CzWindowDataset(reference=REFERENCE, cells=all_cells[:50], chrom=CHROM,
                         binsize=BINSIZE, index=CGN_INDEX, dtype=DTYPE, jobs=16,
                         to_torch=True)
    for w in ds:
        print("torch tensor:", type(w.mc).__name__, tuple(w.mc.shape), w.mc.dtype)
        break
    ds.close()
except ImportError:
    print("torch not installed, skipping")

torch tensor: Tensor (50, 266) torch.uint8


## 7. Close the loader

In [15]:
loader.close()
print("done")

done
